<a href="https://colab.research.google.com/github/tomaszwienke-lgtm/learning-git-task/blob/master/Zadanie_statystyki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza statystyk opisowych na danych z rozkładu normalnego

**Autor:** [Tomasz Wienke]  
**Data:** [15.03.2026]  
**Opis:** W tym notebooku generuję dwie próbki z rozkładu normalnego N(3,1) o liczebnościach 100 i 10000. Dla każdej próbki obliczam podstawowe statystyki opisowe (średnia, mediana, moda, kwartyle, zakres, IQR, wariancja, odchylenie standardowe), przeprowadzam skalowanie zmiennej oraz badam korelację z dodatkowo wygenerowanymi zmiennymi. Celem jest porównanie, jak liczebność próby wpływa na estymację parametrów populacji oraz zrozumienie zachowania poszczególnych miar statystycznych.

## 1. Import bibliotek

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Ustawienie ziarna losowości dla powtarzalności wyników
np.random.seed(42)

## 2. Generowanie danych

Tworzę dwie próbki z rozkładu normalnego o średniej μ = 3 i odchyleniu standardowym σ = 1:
- próbka mała: 100 obserwacji
- próbka duża: 10 000 obserwacji

In [ ]:
n_small = 100
n_large = 10000
mu = 3
sigma = 1

data_small = np.random.normal(mu, sigma, n_small)
data_large = np.random.normal(mu, sigma, n_large)

## 3. Definicja funkcji obliczającej statystyki opisowe

Funkcja `oblicz_statystyki` przyjmuje tablicę danych i nazwę próbki, a następnie wyświetla i zwraca słownik z następującymi miarami:
- Średnia (mean)
- Mediana (median)
- Moda (mode) – dla danych ciągłych zaokrąglam do 2 miejsc, aby uzyskać sensowną wartość
- Kwartyle (Q1, Q3)
- Rozstęp międzykwartylowy (IQR)
- Minimum, maksimum i zakres
- Wariancja próbkowa (z ddof=1)
- Odchylenie standardowe próbkowe

In [ ]:
from collections import Counter  # dodajemy import na górze komórki

def oblicz_statystyki(dane, nazwa):
    """
    Funkcja oblicza podstawowe statystyki opisowe dla próbki.
    Parametry:
        dane (array): wektor danych
        nazwa (str): etykieta próbki (np. "małej")
    Zwraca:
        słownik z obliczonymi statystykami
    """
    print(f"\n--- Statystyki dla próbki {nazwa} (n={len(dane)}) ---")

    srednia = np.mean(dane)
    mediana = np.median(dane)

    # Moda – zaokrąglamy do 2 miejsc i szukamy najczęstszej wartości
    rounded = np.round(dane, 2)
    counter = Counter(rounded)
    # most_common(1) zwraca listę [(wartość, liczba)]
    moda = counter.most_common(1)[0][0]

    Q1 = np.percentile(dane, 25)
    Q3 = np.percentile(dane, 75)
    IQR = Q3 - Q1

    min_val = np.min(dane)
    max_val = np.max(dane)
    zakres = max_val - min_val

    wariancja = np.var(dane, ddof=1)
    odch_std = np.std(dane, ddof=1)

    # Wyświetlanie wyników
    print(f"Średnia: {srednia:.4f}")
    print(f"Mediana: {mediana:.4f}")
    print(f"Moda (po zaokr. do 2 miejsc): {moda}")
    print(f"Q1: {Q1:.4f}, Q3: {Q3:.4f}")
    print(f"IQR: {IQR:.4f}")
    print(f"Min: {min_val:.4f}, Max: {max_val:.4f}, Zakres: {zakres:.4f}")
    print(f"Wariancja (próbkowa): {wariancja:.4f}")
    print(f"Odchylenie standardowe: {odch_std:.4f}")

    return {
        'średnia': srednia,
        'mediana': mediana,
        'moda': moda,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'min': min_val,
        'max': max_val,
        'zakres': zakres,
        'wariancja': wariancja,
        'odch_std': odch_std
    }

## 4. Obliczenie statystyk dla obu próbek

In [ ]:
# Wywołanie funkcji – dopiero teraz zobaczysz wyniki!
stats_small = oblicz_statystyki(data_small, "małej")
stats_large = oblicz_statystyki(data_large, "dużej")

## 5. Skalowanie zmiennej (standaryzacja)

Standaryzacja polega na odjęciu średniej i podzieleniu przez odchylenie standardowe. Po takim przekształceniu nowa zmienna powinna mieć średnią 0 i odchylenie standardowe 1 (w granicach błędu dla próbki).

In [ ]:
data_small_std = (data_small - np.mean(data_small)) / np.std(data_small, ddof=1)
data_large_std = (data_large - np.mean(data_large)) / np.std(data_large, ddof=1)

print("\n--- Po standaryzacji (próbka mała) ---")
print(f"Średnia: {np.mean(data_small_std):.4f}, Odch. std.: {np.std(data_small_std, ddof=1):.4f}")

print("--- Po standaryzacji (próbka duża) ---")
print(f"Średnia: {np.mean(data_large_std):.4f}, Odch. std.: {np.std(data_large_std, ddof=1):.4f}")

## 6. Analiza korelacji

Aby zbadać korelację, tworzę dwie dodatkowe zmienne:
- **Y = X + ε** – zmienna celowo skorelowana z X (gdzie ε to szum normalny o σ = 0.5)
- **Z** – zmienna niezależna, również z rozkładu N(3,1)

Obliczam współczynnik korelacji liniowej Pearsona między X a Y oraz X a Z dla obu próbek. Dla dużej próbki korelacja z Y powinna być wyraźnie dodatnia, a z Z – bliska zeru.

In [ ]:
# Dla małej próbki
epsilon_small = np.random.normal(0, 0.5, n_small)
Y_small = data_small + epsilon_small
Z_small = np.random.normal(3, 1, n_small)

corr_X_Y_small = np.corrcoef(data_small, Y_small)[0,1]
corr_X_Z_small = np.corrcoef(data_small, Z_small)[0,1]

print("\n--- Korelacja (próbka mała) ---")
print(f"Korelacja X z Y (skorelowane): {corr_X_Y_small:.4f}")
print(f"Korelacja X z Z (niezależne): {corr_X_Z_small:.4f}")

# Dla dużej próbki
epsilon_large = np.random.normal(0, 0.5, n_large)
Y_large = data_large + epsilon_large
Z_large = np.random.normal(3, 1, n_large)

corr_X_Y_large = np.corrcoef(data_large, Y_large)[0,1]
corr_X_Z_large = np.corrcoef(data_large, Z_large)[0,1]

print("\n--- Korelacja (próbka duża) ---")
print(f"Korelacja X z Y (skorelowane): {corr_X_Y_large:.4f}")
print(f"Korelacja X z Z (niezależne): {corr_X_Z_large:.4f}")

## 7. Wizualizacja wyników

Przygotowuję cztery wykresy:
1. Histogram dla małej próbki z zaznaczoną średnią i medianą.
2. Histogram dla dużej próbki z analogicznymi oznaczeniami.
3. Boxplot porównawczy obu próbek – pokazuje rozkład, kwartyle, wartości odstające.
4. Wykres rozrzutu X vs Y dla dużej próbki (pierwsze 500 punktów dla czytelności) z wartością współczynnika korelacji.

In [ ]:
plt.figure(figsize=(14, 10))

# Histogram - mała próbka
plt.subplot(2, 2, 1)
sns.histplot(data_small, kde=True, bins=15, color='skyblue')
plt.axvline(stats_small['średnia'], color='red', linestyle='--', label=f"Średnia = {stats_small['średnia']:.2f}")
plt.axvline(stats_small['mediana'], color='green', linestyle=':', label=f"Mediana = {stats_small['mediana']:.2f}")
plt.title(f'Histogram - próbka mała (n={n_small})')
plt.legend()

# Histogram - duża próbka
plt.subplot(2, 2, 2)
sns.histplot(data_large, kde=True, bins=30, color='lightcoral')
plt.axvline(stats_large['średnia'], color='red', linestyle='--', label=f"Średnia = {stats_large['średnia']:.2f}")
plt.axvline(stats_large['mediana'], color='green', linestyle=':', label=f"Mediana = {stats_large['mediana']:.2f}")
plt.title(f'Histogram - próbka duża (n={n_large})')
plt.legend()

# Boxplot porównawczy
plt.subplot(2, 2, 3)
data_box = [data_small, data_large]
plt.boxplot(data_box, labels=['Mała (n=100)', 'Duża (n=10000)'])
plt.title('Boxplot porównawczy')
plt.ylabel('Wartość')

# Wykres korelacji X vs Y (duża próbka)
plt.subplot(2, 2, 4)
plt.scatter(data_large[:500], Y_large[:500], alpha=0.6, s=10, c='purple')
plt.title(f'X vs Y (skorelowane), r = {corr_X_Y_large:.3f}')
plt.xlabel('X')
plt.ylabel('Y = X + szum')

plt.tight_layout()
plt.show()

## 8. Wnioski

Po przeanalizowaniu wyników dla próbek 100 i 10000 obserwacji mogę wyciągnąć następujące wnioski:

- **Średnia i mediana** – w obu przypadkach są bliskie prawdziwej średniej populacji (3), ale dla próbki 10000 są praktycznie równe 3 (błąd estymacji jest mniejszy). Potwierdza to prawo wielkich liczb.
- **Moda** – dla danych ciągłych moda nie jest jednoznaczna; po zaokrągleniu uzyskujemy wartość bliską średniej, ale dla małej próbki może być bardziej przypadkowa.
- **Kwartyle i IQR** – wartości te są stabilne i podobne dla obu próbek, co świadczy o tym, że rozkład jest symetryczny.
- **Zakres (min, max)** – w większej próbce wartości skrajne mogą być nieco bardziej oddalone od średniej, ponieważ większa liczebność zwiększa szansę na pojawienie się ekstremalnych obserwacji.
- **Wariancja i odchylenie standardowe** – dla dużej próbki są bardzo bliskie 1 (prawdziwa wariancja populacji), podczas gdy dla małej próbki mogą nieco odbiegać.
- **Standaryzacja** – po przekształceniu obie próbki mają średnią bliską 0 i odchylenie standardowe bliskie 1. Dla małej próbki te wartości są obarczone większym błędem.
- **Korelacja** – dla zmiennej celowo skorelowanej (Y = X + szum) współczynnik korelacji jest wysoki (~0,9) i stabilny niezależnie od wielkości próby. Dla zmiennych niezależnych (X i Z) korelacja jest bliska zeru, a w większej próbce jeszcze bliższa zera, co jest zgodne z oczekiwaniami.

**Ogólna konkluzja:** Większa próbka dostarcza bardziej precyzyjnych estymatorów parametrów populacji. Dla celów analizy danych zawsze warto dążyć do jak największej próby, pamiętając jednak o kosztach i możliwościach jej pozyskania. Wizualizacje (histogramy, boxploty) pomagają ocenić kształt rozkładu i ewentualną obecność wartości odstających.

## 9. Zapis do pliku (opcjonalnie)

In [ ]:
# Przykład zapisu danych do CSV
df_small = pd.DataFrame({'X_small': data_small, 'Y_small': Y_small, 'Z_small': Z_small})
df_large = pd.DataFrame({'X_large': data_large[:1000], 'Y_large': Y_large[:1000], 'Z_large': Z_large[:1000]})  # tylko 1000 dla oszczędności
df_small.to_csv('dane_mala_proba.csv', index=False)
df_large.to_csv('dane_duza_proba_sample.csv', index=False)
print("Pliki zapisane.")